# 03 — Evaluación del modelo

Evaluación cuantitativa (ROUGE-L, BLEU) y cualitativa (human eval) del modelo fine-tuneado.

In [ ]:
!pip install -q evaluate rouge-score sacrebleu transformers

In [ ]:
import json
import torch
import evaluate
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, pipeline

MODEL_ID = 'lopezinsua/mistral-7b-code-reviewer-es'
DATA_PATH = '../data/dataset.jsonl'

rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')

## 1. Cargar modelo fine-tuneado

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pipe = pipeline(
    'text-generation',
    model=MODEL_ID,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map='auto',
)
print('Modelo cargado')

## 2. Cargar test set

In [ ]:
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

# Usar los últimos 10% como test (mismo split que entrenamiento)
n_test = max(1, int(len(records) * 0.1))
test_records = records[-n_test:]
print(f"Test set: {len(test_records)} ejemplos")

## 3. Generar predicciones

In [ ]:
def predict(record: dict, max_new_tokens: int = 256) -> str:
    prompt = f"<s>[INST] {record['instruction']}\n\n{record['input']} [/INST]"
    result = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    generated = result[0]['generated_text']
    return generated.split('[/INST]')[-1].strip().replace('</s>', '').strip()

predictions, references = [], []
for r in test_records:
    pred = predict(r)
    predictions.append(pred)
    references.append(r['output'])

print(f"Predicciones generadas: {len(predictions)}")

## 4. Métricas automáticas

In [ ]:
rouge_scores = rouge_metric.compute(predictions=predictions, references=references)
bleu_score = bleu_metric.compute(
    predictions=predictions,
    references=[[r] for r in references],
)

results = {
    'ROUGE-1': round(rouge_scores['rouge1'], 4),
    'ROUGE-2': round(rouge_scores['rouge2'], 4),
    'ROUGE-L': round(rouge_scores['rougeL'], 4),
    'BLEU': round(bleu_score['score'], 4),
}

for metric, value in results.items():
    print(f"{metric}: {value}")

## 5. Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results.keys(), results.values(), color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.set_ylim(0, 1)
ax.set_title('Métricas de evaluación — mistral-7b-code-reviewer-es')
ax.set_ylabel('Score')
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}', ha='center')
plt.tight_layout()
plt.savefig('../outputs/eval_metrics.png', dpi=150)
plt.show()

## 6. Human evaluation (manual)

Evalúa 20-25 ejemplos manualmente. Escala: 1-5 (1=incorrecto, 5=excelente)

In [ ]:
# Mostrar ejemplos para evaluación manual
for i, (r, pred) in enumerate(zip(test_records[:5], predictions[:5])):
    print(f"\n{'='*60}")
    print(f"Ejemplo {i+1}")
    print(f"INPUT:\n{r['input']}")
    print(f"\nREFERENCIA:\n{r['output']}")
    print(f"\nPREDICCIÓN:\n{pred}")
    print()

In [ ]:
# Rellenar tras evaluación manual
human_eval = {
    'n_evaluated': 25,
    'avg_score': None,   # reemplazar con promedio 1-5
    'notes': '',
}

import json, os
os.makedirs('../outputs', exist_ok=True)
with open('../outputs/eval_results.json', 'w') as f:
    json.dump({**results, 'human_eval': human_eval}, f, indent=2)
print('Resultados guardados en outputs/eval_results.json')